# Current exact-Hessian CUDA-graph capture diagnostic

This standalone Colab test settles whether the **current combined Hessian** can be manually captured after every host transfer and synchronization has been moved outside the graph body.

Each `(piece, capture_error_mode)` attempt runs in a fresh subprocess because one invalid capture can poison its CUDA context. Every subprocess:

1. builds the production one-day collocation workload and records real IPOPT Hessian inputs;
2. verifies a matrix-multiplication capture canary;
3. enables CUDA synchronization warnings;
4. warms the device-only closure on a side stream;
5. captures and replays it;
6. compares replay against eager and reports the complete traceback on failure.

Pieces:

- `curvature`: only `vmap(torch.func.hessian(...))` for the current combined scalar Lagrangian;
- `body`: curvature plus multiplier assembly and sparse-Hessian packing.

Run on a GPU Colab runtime.

In [ ]:
# --- Install the branch and prepare isolated runner files --------------------
import json
from pathlib import Path
import shutil
import subprocess
import sys

import torch

REF = "feature/issue-124/fuse-collocation-callbacks"
REPO_URL = "https://github.com/JBjoernskov/Twin4Build.git"
ROOT = Path("/content/t4b_hessian_graph_diagnostic")

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA Colab runtime is required.")

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--no-cache-dir",
        f"git+{REPO_URL}@{REF}",
    ],
    check=True,
)
shutil.rmtree(ROOT, ignore_errors=True)
ROOT.mkdir(parents=True)

props = torch.cuda.get_device_properties(0)
print(f"GPU: {props.name} ({props.total_memory / 1e9:.1f} GB)")
print(f"PyTorch: {torch.__version__}")

In [ ]:
# --- Write the fresh-process capture runner ---------------------------------
RUNNER = ROOT / "capture_arm.py"
RUNNER.write_text(r'''
import datetime
import importlib.util
import json
import os
from pathlib import Path
import sys
import traceback

import torch
from torch.utils._pytree import tree_flatten
from dateutil import tz

piece, mode, output_path = sys.argv[1], sys.argv[2], Path(sys.argv[3])
os.environ["TWIN4BUILD_GRAPH_DEBUG"] = "1"

import twin4build as tb
import twin4build.estimator._transcription as transcription
import twin4build.examples as examples_package
import twin4build.examples.utils as example_utils

example_path = Path(examples_package.__file__).parent / "full_workflow_example.py"
spec = importlib.util.spec_from_file_location("_graph_diagnostic_workflow", example_path)
workflow = importlib.util.module_from_spec(spec)
spec.loader.exec_module(workflow)


def build_model():
    model = tb.Model(id=f"graph_{piece}_{mode}")
    model.load(
        semantic_model_filename=example_utils.get_path(
            ["estimator_example", "one_room_example_model.xlsm"]
        ),
        fcn=workflow.fcn,
    )
    model.to("cuda", torch.float64)
    return model


def build_parameters(model):
    c = model.components
    space, heater = c["office"], c["office_space_heater"]
    hc = c["office_temperature_heating_controller"]
    cc = c["office_co2_controller"]
    valve = c["office_space_heater_valve"]
    sup, exh = c["office_supply_damper"], c["office_exhaust_damper"]
    occ, wall = c["office_occupancy"], c["office_boundary_wall"]
    det = c["office_occupancy_detector"]
    return [
        (space, "thermal.C_air", 5e5, 1e4, 5e5),
        (space, "thermal.C_wall", 1e6, 1e5, 3e6),
        (wall, "C", 1e6, 1e4, 1e7),
        (space, "thermal.R_out", 0.5, 0.01, 1),
        (space, "thermal.R_in", 0.1, 0.01, 1),
        (wall, "R_a", 0.04, 1e-4, 1),
        (wall, "R_b", 0.04, 1e-4, 1),
        (space, "thermal.f_wall", 0.1, 0, 10),
        (space, "thermal.f_air", 0.1, 0, 10),
        (space, "thermal.Q_occ_gain", 100.0, 10, 200),
        (heater, "thermalMassHeatCapacity", 1e4, 1e3, 2e5),
        (heater, "UA", None, 1, 100),
        (hc, "kp", 0.005, 1e-5, 1, "private"),
        (cc, "kp", 0.0001, 1e-5, 1, "private"),
        ([hc, cc], "Ti", 30, 1, 300, "private"),
        ([hc, cc], "Td", 0, 0, 1, "private"),
        (valve, "waterFlowRateMax", 0.001, 1e-6, 0.1),
        (valve, "valveAuthority", 1, 0.4, 1),
        ([sup, occ.supply_damper], "a", 1, 1, 10, "shared"),
        ([sup, occ.supply_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([exh, occ.exhaust_damper], "a", 1, 1, 10, "shared"),
        ([exh, occ.exhaust_damper], "nominalAirFlowRate", 0.1, 1e-5, 1, "shared"),
        ([space, occ], "mass.V", 65, 50, 80, "shared"),
        ([space, occ], "mass.G_occ", 1e-6, 1e-6, 1e-5, "shared"),
        ([space, occ], "mass.m_inf", 0.001, 1e-4, 0.01, "shared"),
        (det, "threshold", 1.0, 0.02, 5.0),
    ]


def build_measurements(model):
    c = model.components
    return [
        (c["office_valve_position_sensor"], 0.05 / 2),
        (c["office_temperature_sensor"], 0.1 / 2),
        (c["office_damper_position_sensor"], 0.05 / 2),
        (c["office_co2_sensor"], 30 / 2),
    ]


def canary():
    a = torch.randn((32, 32), dtype=torch.float64, device="cuda")
    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph):
        out = a @ a
    graph.replay()
    torch.cuda.synchronize()
    return bool(torch.isfinite(out).all())


record = {
    "piece": piece,
    "mode": mode,
    "torch": torch.__version__,
    "gpu": torch.cuda.get_device_name(0),
}
try:
    model = build_model()
    estimator = tb.Estimator(tb.Simulator(model))
    start = datetime.datetime(2023, 12, 2, tzinfo=tz.gettz("Europe/Copenhagen"))
    estimator.estimate(
        parameters=build_parameters(model),
        measurements=build_measurements(model),
        start_time=[start],
        end_time=[start + datetime.timedelta(hours=24)],
        step_size=1200,
        n_warmup=20,
        method=("casadi", "ipopt", "ad", "collocation"),
        options={
            "maxiter": 2,
            "exact_hessian": True,
            "early_stopping": False,
            "boundary_state_init": "rollout",
        },
    )
    debug = transcription._CUDA_GRAPH_DEBUG
    inputs = debug["inputs"]
    if piece == "curvature":
        fn = debug["combined_curvature"]
        kwargs = {
            "theta_norm": inputs["theta_norm"],
            "y_norm": inputs["y_norm"],
            "lam_by_seg": inputs["lam_by_seg"],
            "s_gn": inputs["s_gn"],
        }
    elif piece == "body":
        fn = debug["hess_device_body"]
        kwargs = {
            "theta_norm": inputs["theta_norm"],
            "y_norm": inputs["y_norm"],
            "lam_mat": inputs["lam_mat"],
            "s_gn": inputs["s_gn"],
        }
    else:
        raise ValueError(f"unknown piece: {piece}")

    kwargs = {name: value.detach().clone() for name, value in kwargs.items()}
    record["canary"] = canary()
    if not record["canary"]:
        raise RuntimeError("capture canary failed before the real attempt")

    torch.cuda.set_sync_debug_mode("warn")
    side = torch.cuda.Stream()
    side.wait_stream(torch.cuda.current_stream())
    with torch.cuda.stream(side):
        for _ in range(3):
            fn(**kwargs)
    torch.cuda.current_stream().wait_stream(side)
    torch.cuda.synchronize()

    eager = fn(**kwargs)
    eager_leaves, eager_spec = tree_flatten(eager)
    eager_leaves = [leaf.detach().clone() for leaf in eager_leaves]
    torch.cuda.synchronize()

    graph = torch.cuda.CUDAGraph()
    with torch.cuda.graph(graph, capture_error_mode=mode):
        captured = fn(**kwargs)
    captured_leaves, captured_spec = tree_flatten(captured)
    graph.replay()
    torch.cuda.synchronize()

    if captured_spec != eager_spec or len(captured_leaves) != len(eager_leaves):
        raise RuntimeError("captured output structure differs from eager")
    deltas = [
        float((got - expected).abs().max())
        for got, expected in zip(captured_leaves, eager_leaves)
    ]
    record.update(
        captured=True,
        replay_equal=all(torch.equal(got, expected) for got, expected in zip(captured_leaves, eager_leaves)),
        max_abs_delta=max(deltas, default=0.0),
        output_tensors=len(captured_leaves),
    )
except Exception as exc:
    record.update(
        captured=False,
        error_type=type(exc).__name__,
        error=str(exc),
        traceback=traceback.format_exc(),
    )

output_path.write_text(json.dumps(record, indent=2))
print(json.dumps(record, indent=2))
''')
print(f"Wrote {RUNNER}")

In [ ]:
# --- Run every capture attempt in a clean CUDA process ----------------------
import os
import pandas as pd
from IPython.display import display

arms = [
    (piece, mode)
    for piece in ("curvature", "body")
    for mode in ("global", "relaxed")
]
rows = []
for piece, mode in arms:
    print(f"Running {piece} / {mode} ...", flush=True)
    output = ROOT / f"{piece}_{mode}.json"
    proc = subprocess.run(
        [sys.executable, str(RUNNER), piece, mode, str(output)],
        text=True,
        capture_output=True,
        env={**os.environ, "PYTHONUNBUFFERED": "1"},
    )
    if not output.exists():
        raise RuntimeError(
            f"Runner crashed before producing a report ({piece}/{mode}).\n"
            f"STDOUT:\n{proc.stdout}\nSTDERR:\n{proc.stderr}"
        )
    row = json.loads(output.read_text())
    row["returncode"] = proc.returncode
    row["sync_warnings"] = "\n".join(
        line for line in proc.stderr.splitlines()
        if "synchron" in line.lower() or "captur" in line.lower()
    )
    rows.append(row)

summary = pd.DataFrame(
    [
        {
            key: row.get(key)
            for key in (
                "piece",
                "mode",
                "canary",
                "captured",
                "replay_equal",
                "max_abs_delta",
                "output_tensors",
                "error_type",
                "error",
            )
        }
        for row in rows
    ]
)
display(summary)

for row in rows:
    print("\n" + "=" * 90)
    print(f"{row['piece']} / {row['mode']}")
    if row.get("sync_warnings"):
        print("Synchronization/capture warnings:")
        print(row["sync_warnings"])
    if not row.get("captured"):
        print("Full traceback:")
        print(row.get("traceback", "<missing>"))

if summary["captured"].all() and summary["replay_equal"].all():
    print("\nVERDICT: all current combined-Hessian regions capture and replay exactly.")
elif not summary["captured"].any():
    print("\nVERDICT: no current combined-Hessian region captured; inspect the first traceback above.")
else:
    print("\nVERDICT: capture support is partial; the table localizes the failing boundary.")